# Module 05 — Exercise: Damage Assessment Methodologies

**DIGIHAZ PhD Course** | Disaster Risk Reduction


## Student Information

- **Name:** Yue Qimin
- **Student ID:** Not provided
- **Date:** 2026-10-01


## 1. Optical vs. SAR disaster-response workflows

| Aspect | Optical | SAR |
|---|---|---|
| Measurement | Passive reflected solar radiation | Active microwave backscatter and phase |
| Strengths | Intuitive spectral/visual interpretation; useful for vegetation, water and surface-material changes | Day/night imaging and cloud/smoke penetration; sensitive to roughness, moisture, geometry and structural disturbance |
| Limitations | Clouds, haze, smoke and illumination | Speckle, layover, shadow, incidence-angle effects and decorrelation |
| Typical change indicators | NDVI/NDWI/NBR change, spectral difference, classification | Backscatter ratio/log-ratio, coherence loss, texture change |
| Key preprocessing | Cloud masking, atmospheric correction, co-registration | Calibration, co-registration, speckle handling, terrain correction |

A practical workflow is: **define event → select pre/post imagery → preprocess → co-register → compute change → apply contextual masks → classify → validate → communicate uncertainty**.


## 2. San Vicente, Dangcagan, Bukidnon case interpretation

The lecture identifies interacting contributors rather than a single cause:
1. denuded forest / reduced vegetation cover;
2. conversion to agriculture and urbanized land; and
3. extreme rainfall combined with soft soil (Kidapawan Clay).

This shows why satellite change detection should be interpreted together with terrain, land use, soil information and field observations.


## 3. Reproducible NDVI-drop + slope-mask example

The following synthetic example demonstrates the lecture logic. It is a method demonstration, not a real disaster product.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
rows, cols = 80, 80

red_pre = np.clip(rng.normal(0.18, 0.025, (rows, cols)), 0.05, 0.40)
nir_pre = np.clip(rng.normal(0.58, 0.035, (rows, cols)), 0.15, 0.85)

red_post = red_pre.copy()
nir_post = nir_pre.copy()

true_landslide = np.zeros((rows, cols), dtype=bool)
true_landslide[28:57, 33:62] = True
red_post[true_landslide] = np.clip(red_post[true_landslide] + 0.16, 0, 1)
nir_post[true_landslide] = np.clip(nir_post[true_landslide] - 0.30, 0, 1)

harvest = np.zeros((rows, cols), dtype=bool)
harvest[8:22, 8:28] = True
red_post[harvest] = np.clip(red_post[harvest] + 0.10, 0, 1)
nir_post[harvest] = np.clip(nir_post[harvest] - 0.20, 0, 1)

slope = np.clip(rng.normal(12, 4, (rows, cols)), 0, 35)
slope[22:65, 25:68] += 12
slope = np.clip(slope, 0, 45)

def ndvi(nir, red):
    return (nir - red) / (nir + red + 1e-9)

ndvi_pre = ndvi(nir_pre, red_pre)
ndvi_post = ndvi(nir_post, red_post)
delta_ndvi = ndvi_post - ndvi_pre

candidate = delta_ndvi <= -0.25
detected = candidate & (slope >= 15)

pixel_size_m = 10
detected_area_ha = detected.sum() * pixel_size_m**2 / 10000

pd.DataFrame({
    "Metric":["NDVI-drop candidate pixels","Pixels after slope mask","Detected area (ha)"],
    "Value":[int(candidate.sum()), int(detected.sum()), round(detected_area_ha,2)]
})


In [ ]:
fig, axes = plt.subplots(2,2,figsize=(10,8))
axes[0,0].imshow(ndvi_pre, cmap="RdYlGn", vmin=-1, vmax=1); axes[0,0].set_title("Pre-event NDVI")
axes[0,1].imshow(delta_ndvi, cmap="RdBu", vmin=-0.7, vmax=0.7); axes[0,1].set_title("NDVI change")
axes[1,0].imshow(slope, cmap="terrain", vmin=0, vmax=45); axes[1,0].set_title("Slope (degrees)")
axes[1,1].imshow(detected, cmap="gray"); axes[1,1].set_title("Landslide proxy")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Interpretation

The slope mask reduces false positives from vegetation change on flat terrain. However, NDVI change alone does not prove a landslide; cloud contamination, phenology, harvesting, development and registration errors can also cause apparent change. The result should therefore be validated with independent imagery or field evidence.


## Conclusion

The strongest damage-assessment workflow combines **multi-sensor change detection, contextual geospatial information and independent validation**.
